In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import optuna
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import root_mean_squared_error

In [2]:
data = pd.read_csv("datasets/Exam_Score_Prediction.csv")

X = data.drop('exam_score', axis=1)
y = data['exam_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, shuffle=True, random_state=42)

In [3]:
class AddStudySleepRatio(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        X['study_sleep_ratio'] = X['study_hours'] / X['sleep_hours']
        return X

preprocessing_pipeline = joblib.load('models_and_pipelines/preprocessing_pipeline.pkl')

In [4]:
X_train = preprocessing_pipeline.transform(X_train)
X_val = preprocessing_pipeline.transform(X_val)
X_test = preprocessing_pipeline.transform(X_test)

In [13]:
def objective(trial, model_name):
    if model_name == "Linear Regression": params = {}

    elif model_name == "Lasso": params = {
        "alpha": trial.suggest_float("alpha", 0, 10, step=0.1),
        "tol": trial.suggest_float("tol", 0, 1, step=0.001)
    }

    elif model_name == "Ridge": params = {
        "alpha": trial.suggest_float("alpha", 0, 10, step=0.1),
        "tol": trial.suggest_float("tol", 0, 1, step=0.001)
    }

    elif model_name == "K-Nearest Neighbors": params = {
        "n_neighbors": trial.suggest_int("n_neighbors", 1, 100),
        "leaf_size": trial.suggest_int("leaf_size", 10, 1000, step=10)
    }

    elif model_name == "Support Vector Regression": params = {
        "degree": trial.suggest_int("degree", 1, 10),
        "tol": trial.suggest_float("tol", 0, 1, step=0.001),
        "C": trial.suggest_float("C", 1, 1000, step=0.5),
        "epsilon": trial.suggest_float("epsilon", 0.1, 100, step=0.1),
    }

    elif model_name == "Decision Tree Regression": params = {
        "max_depth": trial.suggest_int("max_depth", 1, 100),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 2, 1000),
    }

    elif model_name == "Random Forest Regression": params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, 10),
        "max_depth": trial.suggest_int("max_depth", 1, 100),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 2, 1000),
    }

    elif model_name == "CatBoosting Regression": params = {
        "iterations": trial.suggest_int("iterations", 1, 501, step=10),
        "learning_rate": trial.suggest_float("learning_rate", 0.05, 0.5, step=0.05),
        "depth": trial.suggest_int("depth", 1, 10),
    }

    elif model_name == "XGBusting Regression": params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.05, 0.5, step=0.05),
        "max_depth": trial.suggest_int("max_depth", 1, 100),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 15),
        "gamma": trial.suggest_float("gamma", 0, 1, step=0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0, 1, step=0.1),
    }

    model = models[model_name]
    model.set_params(**params)

    score = cross_val_score(model, X_train, y_train, cv=3, n_jobs=-1, scoring="neg_root_mean_squared_error")
    final_score = abs(score.mean())

    return final_score
    
models = {
        "Linear Regression": LinearRegression(n_jobs=-1),
        "Lasso": Lasso(), 
        "Ridge": Ridge(),
        "K-Nearest Neighbors": KNeighborsRegressor(n_jobs=-1),
        "Support Vector Regression": SVR(max_iter=10000),
        "Decision Tree Regression": DecisionTreeRegressor(),
        "Random Forest Regression": RandomForestRegressor(n_jobs=-1),
        "CatBoosting Regression": CatBoostRegressor(),
        "XGBusting Regression": XGBRegressor()
        }

models_test_score = {}
models_best_params = {}

for model_name in models.keys():
    study = optuna.create_study(direction="minimize")
    study.optimize(lambda trial: objective(trial, model_name), n_trials=100)

    models_test_score[model_name] = study.best_trial.values[0]
    models_best_params[model_name] = study.best_params

[I 2026-02-21 23:22:57,825] A new study created in memory with name: no-name-8cfb8ced-9549-41cf-b448-b8876a9f5c86
[I 2026-02-21 23:22:57,961] Trial 0 finished with value: 9.802460898402996 and parameters: {}. Best is trial 0 with value: 9.802460898402996.
[I 2026-02-21 23:22:58,096] Trial 1 finished with value: 9.802460898402996 and parameters: {}. Best is trial 0 with value: 9.802460898402996.
[I 2026-02-21 23:22:58,232] Trial 2 finished with value: 9.802460898402996 and parameters: {}. Best is trial 0 with value: 9.802460898402996.
[I 2026-02-21 23:22:58,381] Trial 3 finished with value: 9.802460898402996 and parameters: {}. Best is trial 0 with value: 9.802460898402996.
[I 2026-02-21 23:22:58,522] Trial 4 finished with value: 9.802460898402996 and parameters: {}. Best is trial 0 with value: 9.802460898402996.
[I 2026-02-21 23:22:58,661] Trial 5 finished with value: 9.802460898402996 and parameters: {}. Best is trial 0 with value: 9.802460898402996.
[I 2026-02-21 23:22:58,807] Trial 

In [ ]:
models_val_score = {}
best_models = {}

for model_name in models.keys():
    model = models[model_name]
    params = models_best_params[model_name]
    model.set_params(**params)
    model.fit(X_test, y_test)

    y_val_pred = model.predict(X_val)
    models_val_score[model_name] = root_mean_squared_error(y_val, y_val_pred)
    best_models[model_name] = model

0:	learn: 14.3916454	total: 149ms	remaining: 58.2s
1:	learn: 12.5056860	total: 156ms	remaining: 30.3s
2:	learn: 11.2634195	total: 162ms	remaining: 20.9s
3:	learn: 10.5703063	total: 166ms	remaining: 16s
4:	learn: 10.2427142	total: 168ms	remaining: 13s
5:	learn: 10.0132399	total: 171ms	remaining: 11s
6:	learn: 9.8783389	total: 173ms	remaining: 9.49s
7:	learn: 9.7692184	total: 175ms	remaining: 8.36s
8:	learn: 9.6847881	total: 176ms	remaining: 7.47s
9:	learn: 9.6445268	total: 177ms	remaining: 6.76s
10:	learn: 9.5852335	total: 178ms	remaining: 6.16s
11:	learn: 9.5315764	total: 180ms	remaining: 5.67s
12:	learn: 9.4948530	total: 181ms	remaining: 5.25s
13:	learn: 9.4548118	total: 182ms	remaining: 4.89s
14:	learn: 9.4062100	total: 183ms	remaining: 4.58s
15:	learn: 9.3597467	total: 184ms	remaining: 4.31s
16:	learn: 9.3201879	total: 185ms	remaining: 4.07s
17:	learn: 9.2698648	total: 186ms	remaining: 3.86s
18:	learn: 9.2430936	total: 187ms	remaining: 3.67s
19:	learn: 9.2004606	total: 188ms	remaini

In [16]:
best_model_name = min(models_val_score, key=models_val_score.get)
best_model = best_models[best_model_name]
best_model_score = models_val_score[best_model_name]

In [18]:
best_model_name, best_model_score

('Ridge', 9.843390013620155)